# BÀI THỰC HÀNH SỐ 1 - KHAI PHÁ DỮ LIỆU
## PHẦN 2 - BÀI 2 (TRANG 37, 38): KHAI PHÁ LUẬT KẾT HỢP DỮ LIỆU ONLINE RETAIL
- **Họ và tên:** Bùi Minh Trọng
- **MSSV:** 6451071081
- **Lớp:** CQ.64.CNTT
- **Tập dữ liệu:** `Online Retail.xlsx`


### Câu 1 (Yêu cầu 1): Đọc dữ liệu vào chương trình


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Họ và tên: Bùi Minh Trọng - MSSV: 6451071081")

import os
data_candidates = ['Online Retail.xlsx', '../TH1/Online Retail.xlsx', '../Online Retail.xlsx', 'datasets/Online Retail.xlsx']
data_path = next((p for p in data_candidates if os.path.exists(p)), 'Online Retail.xlsx')
df_retail = pd.read_excel(data_path)

print(f"Kích thước tập dữ liệu: {df_retail.shape[0]:,} dòng x {df_retail.shape[1]} cột")
display(df_retail.head())
print("\nThông tin cấu trúc tập dữ liệu:")
print(df_retail.info())



**Nhận xét Câu 1:**
- Tập dữ liệu bán lẻ trực tuyến có 541,909 dòng và 8 cột thuộc tính bao gồm: `InvoiceNo`, `StockCode`, `Description`, `Quantity`, `InvoiceDate`, `UnitPrice`, `CustomerID`, `Country`.


### Câu 2 (Yêu cầu 2): Tiền xử lý dữ liệu (Cắt ký tự thừa Description, xóa dòng thiếu InvoiceNo, chuyển InvoiceNo sang string)


In [ ]:
print("Họ và tên: Bùi Minh Trọng - MSSV: 6451071081")

# 1. Cắt bỏ khoảng trắng thừa ở cột Description
df_retail['Description'] = df_retail['Description'].astype(str).str.strip()

# 2. Xóa các dòng dữ liệu không có số hóa đơn InvoiceNo
df_retail.dropna(subset=['InvoiceNo'], inplace=True)

# 3. Chuyển đổi cột InvoiceNo sang kiểu chuỗi (string)
df_retail['InvoiceNo'] = df_retail['InvoiceNo'].astype(str)

print(f"Số dòng sau khi xử lý missing InvoiceNo: {len(df_retail):,}")


**Nhận xét Câu 2:**
- Đã chuẩn hóa mô tả mặt hàng và làm sạch số hóa đơn, đảm bảo tính nhất quán cho các bước gom nhóm giao dịch tiếp theo.


### Câu 3 (Yêu cầu 3): Xóa bỏ các hóa đơn tín dụng (chứa ký tự 'C' trong InvoiceNo)


In [ ]:
print("Họ và tên: Bùi Minh Trọng - MSSV: 6451071081")

# Hóa đơn tín dụng (trả hàng/hủy đơn) có ký tự 'C' trong InvoiceNo
df_clean_retail = df_retail[~df_retail['InvoiceNo'].str.contains('C')].copy()

print(f"Số dòng trước khi xóa hóa đơn 'C': {len(df_retail):,}")
print(f"Số dòng sau khi xóa hóa đơn 'C': {len(df_clean_retail):,}")
print(f"Số lượng hóa đơn tín dụng đã loại bỏ: {len(df_retail) - len(df_clean_retail):,}")


**Nhận xét Câu 3:**
- Đã loại bỏ toàn bộ các giao dịch trả hàng/hủy đơn (hóa đơn tín dụng 'C') nhằm phản ánh chính xác hành vi mua hàng thực tế của người tiêu dùng.


### Câu 4 (Yêu cầu 4): Thống kê số dòng dữ liệu theo từng quốc gia


In [ ]:
print("Họ và tên: Bùi Minh Trọng - MSSV: 6451071081")

country_counts = df_clean_retail['Country'].value_counts()
print("Top 10 quốc gia có số lượng giao dịch nhiều nhất:")
display(country_counts.head(10))

plt.figure(figsize=(10, 4))
country_counts.head(10).plot(kind='bar', color='darkcyan')
plt.title("Top 10 Quốc gia có số lượng giao dịch nhiều nhất")
plt.xlabel("Quốc gia")
plt.ylabel("Số lượng giao dịch")
plt.xticks(rotation=45)
plt.show()


**Nhận xét Câu 4:**
- Nước Anh (United Kingdom) chiếm số lượng giao dịch áp đảo (hơn 480,000 giao dịch, chiếm gần 90% toàn bộ hệ thống). Theo sau là Đức (Germany) và Pháp (France).


### Câu 5 (Yêu cầu 5): Lọc hóa đơn từ United Kingdom và gom nhóm Quantity theo InvoiceNo và Description


In [ ]:
print("Họ và tên: Bùi Minh Trọng - MSSV: 6451071081")

basket_uk = (df_clean_retail[df_clean_retail['Country'] == 'United Kingdom']
             .groupby(['InvoiceNo', 'Description'])['Quantity']
             .sum().unstack().reset_index().fillna(0)
             .set_index('InvoiceNo'))

print("Kích thước ma trận giỏ hàng của United Kingdom:", basket_uk.shape)
display(basket_uk.iloc[:5, :5])


**Nhận xét Câu 5:**
- Đã tạo thành công ma trận giỏ hàng cho thị trường Anh: mỗi hàng là một hóa đơn (`InvoiceNo`), mỗi cột là một mặt hàng (`Description`), giá trị trong ô là tổng số lượng đã mua.


### Câu 6 (Yêu cầu 6, 7, 8, 9, 10): Chuyển sang One-Hot Encoding, loại bỏ 'POSTAGE', tìm tập phổ biến và rút trích luật kết hợp


In [ ]:
print("Họ và tên: Bùi Minh Trọng - MSSV: 6451071081")

# 1. Hàm chuyển đổi hot-encoding sang one-hot (>= 1 thành 1, <= 0 thành 0)
def encode_units(x):
    return 1 if x >= 1 else 0

basket_sets_uk = basket_uk.applymap(encode_units)

# 2. Xóa cột cước phí POSTAGE nếu tồn tại
if 'POSTAGE' in basket_sets_uk.columns:
    basket_sets_uk.drop('POSTAGE', inplace=True, axis=1)

# 3. Áp dụng thuật toán Apriori với min_sup = 3%
from mlxtend.frequent_patterns import apriori, association_rules

frequent_itemsets = apriori(basket_sets_uk, min_support=0.03, use_colnames=True)
print(f"Số lượng tập phổ biến tìm được (min_sup = 3%): {len(frequent_itemsets)}")
display(frequent_itemsets.head(10))

# 4. Rút trích các luật kết hợp với min_conf = 50%
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)
print(f"\nSố lượng luật kết hợp tìm được (min_conf = 50%): {len(rules)}")
display(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))



**Nhận xét Câu 6:**
- Đã chuyển ma trận thành công sang dạng One-Hot 0/1 và loại bỏ cước phí `POSTAGE`.
- Thuật toán Apriori đã tìm được các tập mặt hàng phổ biến với độ hỗ trợ tối thiểu 3%. Các luật kết hợp có độ tin cậy trên 50% và hệ số lift > 1 phản ánh mối quan hệ mua kèm có ý nghĩa thực tiễn rất cao trong kinh doanh bán lẻ.
